In [1]:
import numpy as np
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFDirectoryLoader
from sklearn.metrics.pairwise import cosine_similarity

import dotenv
import os

dotenv.load_dotenv()


True

# Trabajo Práctico 2

Leer CV 

In [2]:
def read_doc(directory):
    file_loader=PyPDFDirectoryLoader(directory)
    documents = file_loader.load()
    return documents

doc_cv=read_doc("docs/")

total=doc_cv

Seccionarlo en chunks de datos

In [3]:
def chunk_data(docs, chunk_size=800, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return text_splitter.split_documents(docs)

documents = chunk_data(docs=total, chunk_size=os.getenv("DOCS_CHUNK_SIZE"), chunk_overlap=os.getenv("DOCS_CHUNK_OVERLAP"))

type(documents)

documents

[Document(metadata={'producer': 'Mac OS X 10.11.1 Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20160506042453Z00'00'", 'title': 'CV Christian Pisani English', 'moddate': "D:20160506042453Z00'00'", 'source': 'docs/CV Christian Pisani English.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='PISANI TESTA \nCHRISTIAN GABRIEL \nExtract\nSoftware Engineer experienced in the full development lifecycle of web \napplications. I have participated in numerous software development projects for \ndifferent areas applying a diverse variety of technologies. I consider myself a fast \nlearner and excellent team player.  \nPersonal Information\n• Birth Date: 05/08/1991 \n• Id: 36.158.170 \n• Address: 6142 Alvarez Jonte Av . - Capital Federal \n• Mobile: (11-54) 15 69 26 23 38 \n• Mail: christian.tpg@gmail.com \nEducation\nUniversity 03/2009 – 12/2014 \n Degree: Software Engineer. \n Institution: National University of Technology in Argentina, FRBA. \n Status:  Finished

Generar los embeddings utilizando hugging face porque groq no tiene generador de embeddings

In [4]:
texts = [doc.page_content for doc in documents]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectors = embeddings.embed_documents(texts)

print("Dimensión del embedding:", len(vectors[0]))
print("Embedding de primer texto (primeros 10 valores):", vectors[0][:10])

Dimensión del embedding: 384
Embedding de primer texto (primeros 10 valores): [-0.07400797307491302, 0.01692265085875988, -0.015124437399208546, 0.01005117129534483, -0.052537646144628525, -0.11054354161024094, 0.013242091983556747, 0.0009151609265245497, -0.08628794550895691, -0.0031017770525068045]


Puedo calcular la similitud coseno entre el embedding de una query y los ya calculados

In [5]:
query = "¿Qué experiencia tiene en machine learning?"
query_vector = embeddings.embed_query(query)

X = np.array(vectors)
q = np.array(query_vector).reshape(1, -1)

sims = cosine_similarity(q, X)[0]

best_idx = int(np.argmax(sims))
best_score = sims[best_idx]

print("Pregunta:", query)
print("Chunk más cercano:", documents[best_idx].page_content)
print("Similitud coseno:", best_score)

Pregunta: ¿Qué experiencia tiene en machine learning?
Chunk más cercano: Part of a team responsible for the development of several web applications in Java. 
Which are destined to internal administration of PAMI (the biggest health insurance 
company in Latin America). 
Projects
Desafíos Mas y Menos 
Mobile Android application that helps children to develop logic and mathematical 
skills with several maths games. The app is written in Java and uses SQLite  
Languages
English  
 Advance - First Certificate in English (FCE) at University of Cambridge. 
Spanish  
  
 Native. 
Skills & Technical Knowledge
Programming Languages Java C# Python C Javascript PL 
SQL Smalltalk
Database: MySQL Oracle 11g SQLServer 
Cassandra SQLite
Operative Systems: Linux iOS Android Windows
Others: Visual Studio Eclipse CSS 
Selenium WebDriver Pentaho ZK 
framework SOAPUI REST UML 
WinForms JUnit HTML GIT
!  2
                     05/05/2016   Page !  of  ! 2 2
Similitud coseno: 0.2750629622034362


Configurar Pinecone

In [6]:
pinecone = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

✅ Pinecone configurado correctamente


Configurar indices

In [7]:
spec = ServerlessSpec(cloud=os.getenv("PINECONE_CLOUD"), region=os.getenv("PINECONE_REGION"))
index_name = os.getenv("PINECONE_INDEX")

if index_name in pinecone.list_indexes().names():
    pinecone.delete_index(index_name)
    print("index {} borrado".format(index_name))

if index_name not in pinecone.list_indexes().names():
    print("index creado con el nombre: {}".format(index_name))
    pinecone.create_index(
        index_name,
        dimension=384,  
        metric='cosine',
        spec=spec
    )
else:
    print("el index con el nombre {} ya estaba creado".format(index_name))

index = pinecone.Index(index_name)

index chrpt borrado
index creado con el nombre: chrpt


Subir los vectores a Pinecone

In [14]:
for i, (doc, vector) in enumerate(zip(documents, vectors)):
    index.upsert([
        (
            f"chunk-{i}",
            vector,
            {"text": doc.page_content}
        )
    ])

Utilizaré Groq como LLM para responder preguntas sobre mi CV. 

In [15]:
groq = ChatGroq(
    model=os.getenv("GROQ_MODEL"),
    temperature=0
)

In [16]:
def ask(llm, query, index=index):
    query_vector = embeddings.embed_query(query)
    
    results = index.query(
        vector=query_vector,
        top_k=3,                # número de chunks relevantes
        include_metadata=True
    )
    
    context = "\n".join([match["metadata"]["text"] for match in results["matches"]])

    prompt = f"""
    Usá la siguiente información de mi CV para responder la pregunta.
    Si no está en el CV, indicá que no se encuentra en la información.
    
    Contexto:
    {context}
    
    Pregunta:
    {query}
    """
    
    print(llm.invoke(prompt).content)

Podré ahora sí hablar con Groq sobre mi experiencia profesional

In [17]:
ask(groq, "¿Qué experiencia tiene en machine learning?")

No se encuentra en la información proporcionada una experiencia específica en machine learning. Sin embargo, se menciona que en su experiencia laboral en Navent (08/2015 - Presente), diseñó y desarrolló aplicaciones de Data Mining en C# utilizando MySQL, Selenium Web Driver, Fiddler y otras tecnologías de crawling web, lo que podría estar relacionado con el análisis y procesamiento de datos, pero no se menciona explícitamente la experiencia en machine learning.


In [18]:
ask(groq, "¿Cuál es el campo profesional al qué se dedicó?")

Según la información proporcionada en tu CV, el campo profesional al que te dedicas es el de Ingeniero de Software, específicamente en el desarrollo de aplicaciones web y móviles, con experiencia en lenguajes de programación como Java, C#, Python, entre otros, y en bases de datos como MySQL, Oracle, SQLServer, Cassandra, SQLite, etc.

En particular, mencionas que has trabajado como Ingeniero de Software en empresas como Navent y Treebeo, y que has participado en proyectos de desarrollo de aplicaciones web y móviles para diferentes áreas, aplicando una variedad de tecnologías.


In [19]:
ask(groq, "¿Cuántos años tiene Christian si estamos en el año 2025?")



Según la información proporcionada en el CV, Christian Gabriel PISANI TESTA nació el 05/08/1991. 

Para calcular su edad en el año 2025, restamos el año de nacimiento del año actual:

2025 - 1991 = 34 años

Entonces, Christian tiene 34 años en el año 2025, considerando que su cumpleaños ya haya pasado en ese año. Si su cumpleaños aún no ha pasado en 2025, tendría 33 años.
